# Supervised analysis (_Iris_)

In [32]:
from experiments.utils.constants import RANDOM_SEED, SOM_LEARNING_RATE_DECAY_FN

VERBOSE = True
DATASET_NAME = "Iris"
DATASET_ID = 53

## Dataset

In [33]:
# fetch dataset
from ucimlrepo import fetch_ucirepo
dataset = fetch_ucirepo(id=DATASET_ID)
X = dataset.data.features.values
y = dataset.data.targets.values.ravel()
print(f"Dataset shape: {X.shape}, {y.shape}")

Dataset shape: (150, 4), (150,)


In [34]:
# scale data
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
del X

In [35]:
# encode labels
import numpy as np
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
y_encoded = le.fit_transform(y)
label_decoder = {i: label for i, label in enumerate(le.classes_)}
num_unique_y = len(np.unique(y_encoded))
print(f"Classes: {num_unique_y}")

Classes: 3


## Re-create model

In [36]:
from minisom_representation import calc_som_hyparams, SomRepresentation

In [37]:
# hyperparameters
recommended_params = calc_som_hyparams(X_scaled, initial_sigma_factor=3.0)
print("Recommended SOM parameters:", recommended_params)
d1, d2, sigma = map(recommended_params.get, ("d1", "d2", "sigma"))
decay_function = SOM_LEARNING_RATE_DECAY_FN
epoch = 50

Recommended SOM parameters: {'d1': 6, 'd2': 11, 'sigma': 3.67}


In [38]:
# fit SOM representation
som_rep = SomRepresentation(d1=d1, d2=d2, sigma=sigma, random_seed=RANDOM_SEED, verbose=VERBOSE, decay_function=decay_function) \
    .fit_online(X_scaled, num_iteration=epoch)

 [ 7500 / 7500 ] 100% - 0:00:00 left 
 quantization error: 0.4326757938872267

 An SOM representation has been fitted as follows:
------------------------------------------------------- 

Fit strategy: online 

Hyperparameters of SOM: 

{'input_len': 4, 'x': 6, 'y': 11, 'sigma': 3.67, 'topology': 'rectangular', 'learning_rate': 0.5, 'decay_function': 'linear_decay_to_zero', 'sigma_decay_function': 'asymptotic_decay', 'neighborhood_function': 'gaussian', 'activation_distance': 'euclidean', 'random_seed': 42, 'num_iteration': 50, 'use_epochs': True, 'random_order': True, 'verbose': True} 

Quality of SOM: 

Quantization Error (QE):	0.4326757938872267
Topographic Error (TE): 	0.0


## Inspection

In [39]:
from utils.plotting import PlotlyHelperArgs

In [40]:
# create Basin
from lilypond import Basin
basin = Basin.from_som_representation(som_rep, random_seed=RANDOM_SEED, verbose=VERBOSE)

In [41]:
# lilypond visual
basin.pond() \
    .rhizome_layer() \
    .pad_layer() \
    .petal_layer() \
    .visualize(width=800, height=400);

## Extra figures

In [42]:
plot_args = dict(
    **PlotlyHelperArgs.Figsize(w=800, h=460),
    **PlotlyHelperArgs.FullStretch,
    **PlotlyHelperArgs.HiddenTicks(d1=d1, d2=d2),
    font=dict(size=35),
    showlegend=False
)

In [43]:
import plotly.express as px
palette = ["#FD3216", "#0DF9FF", "#D626FF"]
marker_colors = [palette[val % len(palette)] for val in y_encoded]

In [44]:
fig1 = basin.pond() \
    .pad_layer(gap="nogap") \
    .rhizome_layer() \
    .attraction_layer(X_scaled, marker=dict(color=marker_colors, symbol="star-diamond", size=20, opacity=.8), name="Projection of training data colored by class") \
    .visualize(**plot_args);

In [45]:
EXPORT_DIR = "./_exports"
fig1.write_image(EXPORT_DIR + "/03_01_lilypond_01.png")
